In [6]:
from langgraph.graph import StateGraph, START, END, MessagesState
from dotenv import load_dotenv, find_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict, Annotated, Literal
from langchain_core.messages import SystemMessage, HumanMessage, BaseMessage
from pydantic import BaseModel, Field
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv(find_dotenv())

True

In [4]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [7]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [8]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [9]:
graph = builder.compile()
graph

In [10]:
graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Parveen."}]})

{'messages': [HumanMessage(content='Hi! My name is Parveen.', id='d708c4d1-f61e-4ed2-8aff-1eeec4fd90c2'),
  AIMessage(content="Hi Parveen! It's nice to meet you.", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-a9bbe8e7-6efa-45bb-9c93-ecbaac0fd71d-0', usage_metadata={'input_tokens': 9, 'output_tokens': 12, 'total_tokens': 44})]}

In [11]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]})

{'messages': [HumanMessage(content='What is my name?', id='9aaf46cf-9de7-4399-a21d-7433d8b8e5ca'),
  AIMessage(content="As an AI, I don't have access to any personal information about you, including your name. I don't retain details from past conversations or have any way of knowing who you are.", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-a7311a4a-7e35-4126-9b32-73b94f6f0b71-0', usage_metadata={'input_tokens': 6, 'output_tokens': 40, 'total_tokens': 400})]}

In [12]:
def call_model(state: MessagesState):
    response = model.invoke(state["messages"])
    return {"messages": [response]}

In [13]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [19]:
config = {"configurable": {"thread_id": "thread-1"}}
# config2 = {"configurable": {"thread_id": "thread-2"}}

In [20]:
graph.invoke({"messages": [{"role": "user", "content": "Hi! My name is Parveen."}]}, config)

{'messages': [HumanMessage(content='Hi! My name is Parveen.', id='71d4406d-836f-4f36-99ac-529549e29be3'),
  AIMessage(content="Hi Parveen! It's nice to meet you.", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-a8187d9f-02b5-42ac-8b36-69a56c871109-0', usage_metadata={'input_tokens': 9, 'output_tokens': 12, 'total_tokens': 48}),
  HumanMessage(content='Hi! My name is Parveen.', id='a85dd836-5b72-49ff-83c3-7870cb89bbae'),
  AIMessage(content="Hello again, Parveen! It's good to hear from you. How can I help you today?", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-9b28f786-b5a1-4884-82b9-cd91536dd0db-0', usage_metadata={'input_tokens': 31, 'output_tokens': 22, 'total_tokens': 265})]}

In [21]:
graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)

{'messages': [HumanMessage(content='Hi! My name is Parveen.', id='71d4406d-836f-4f36-99ac-529549e29be3'),
  AIMessage(content="Hi Parveen! It's nice to meet you.", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-a8187d9f-02b5-42ac-8b36-69a56c871109-0', usage_metadata={'input_tokens': 9, 'output_tokens': 12, 'total_tokens': 48}),
  HumanMessage(content='Hi! My name is Parveen.', id='a85dd836-5b72-49ff-83c3-7870cb89bbae'),
  AIMessage(content="Hello again, Parveen! It's good to hear from you. How can I help you today?", response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-9b28f786-b5a1-4884-82b9-cd91536dd0db-0', usage_metadata={'input_tokens': 31, 'output_tokens': 22, 'total_tokens': 265}),
  HumanMessage(content='What is my name?', id='00558770-aa5d-474b-9ddd-7c7d0b398cd7'),
  AIMessage(content='Your name is Parveen.

In [22]:
snap = graph.get_state(config)
vals = snap.values
for m in vals.get("messages", []):
        print("-", type(m).__name__, ":", m.content)

- HumanMessage : Hi! My name is Parveen.
- AIMessage : Hi Parveen! It's nice to meet you.
- HumanMessage : Hi! My name is Parveen.
- AIMessage : Hello again, Parveen! It's good to hear from you. How can I help you today?
- HumanMessage : What is my name?
- AIMessage : Your name is Parveen.
